## Initialisation

In [ ]:
import pandas as pd


# Fonction pour convertir une colonne en str (pour gérer les colonnes avec des types mixés, qui génèrent des erreurs)
def convert_to_str(df, list_vars):
    for var in list_vars:
        df[var] = df[var].astype(str)
    return df


# Fonction pour compter le nombre de codes communes
def count_geo_codes(df):
    set_cities = set(df['CITY'])
    print(f'Nombre de communes : {len(set_cities)}')
    return set_cities


# Fonction pour transformer les codes en libellés complets en utilisant les métadonnées
def replace_metadata(df, df_meta, list_vars):
    for var in list_vars:
        var_dict = df_meta[df_meta['COD_VAR'] == var].set_index('COD_MOD')['LIB_MOD'].to_dict()
    
        df[var] = df[var].map(var_dict)
    return df


def add_cities_metadata(df, df_meta):
    var_dict = df_meta[df_meta['COD_VAR'] == 'GEO'].set_index('COD_MOD')['LIB_MOD'].to_dict()
    df['CITY'] = df['GEO'].map(var_dict)
    return df


# Fonction pour supprimer les doublons
def handle_duplicates(df):
    duplicates_count = df.duplicated().sum()

    if duplicates_count > 0:
        df = df.drop_duplicates()
        print(f'--- {duplicates_count} doublons supprimés')
    else:
        print("--- Aucun doublon")


# Fonction pour compter les valeurs manquantes
def count_null_values(df):
    null_values_count = df.isnull().sum()
    missing_data = null_values_count[null_values_count > 0]

    if missing_data.empty:
        print('\n--- Aucune valeur manquante')
        return False
    else:
        results = pd.DataFrame({
            'Valeurs manquantes (nb)': missing_data,
            'Valeurs manquantes (%)': round((missing_data / len(df)) * 100, 2)
        })
        print('--- Valeurs manquantes :')
        display(results)
        return True


# Fonction pour traiter les doublons et valeurs manquantes
def handle_null_and_duplicates(df):
    missing_values = count_null_values(df)
    if missing_values:
        print('--- Valeurs manquantes, doublons non traités')
    else:
        handle_duplicates(df)



## Chargement dataset - Population

In [ ]:
# Chargement données + métadonnées
population_csv = pd.read_csv('src_insee/DS_RP_POPULATION_COMP_2022_CSV_FR/DS_RP_POPULATION_COMP_2022_data.csv', sep=';')
df_population = pd.DataFrame(population_csv)


# Suppression des lignes et colonnes inutiles
df_population = df_population[(df_population['TIME_PERIOD'] == 2022) & (df_population['GEO_OBJECT'] == 'COM')]
df_population = df_population.drop(['RP_MEASURE', 'TIME_PERIOD', 'GEO_OBJECT'], axis=1)


# Doublons et valeurs manquantes
print('Doublons et valeurs manquantes après suppression des données inutiles :')
handle_null_and_duplicates(df_population)


# Premier aperçu du dataframe
display(df_population)


# Conversion des colonnes PCS et GEO en str pour éviter les valeurs manquantes générées par les int
df_population = convert_to_str(df_population, ['GEO', 'PCS'])


# Récupération et application des métadonnées
population_meta_csv = pd.read_csv('src_insee/DS_RP_POPULATION_COMP_2022_CSV_FR/DS_RP_POPULATION_COMP_2022_metadata.csv', sep=';')
df_population_meta = pd.DataFrame(population_meta_csv)

df_population = replace_metadata(df_population, df_population_meta, ['AGE', 'SEX', 'PCS'])
df_population = add_cities_metadata(df_population, df_population_meta)


# Doublons et valeurs manquantes après récupération des métadonnées
print('Doublons et valeurs manquantes après récupération des communes :')
handle_null_and_duplicates(df_population)


# Changement de nom pour les colonnes pas claires
df_population.rename(columns={'OBS_VALUE': 'POPULATION', 'PCS': 'SOCIO_PROF_CAT'}, inplace=True)


# Aperçu des données après récupération des métadonnées
display(df_population)

## Chargement dataset - Conjugalité

In [ ]:
# Chargement données + métadonnées
conjugality_csv = pd.read_csv('src_insee/DS_RP_MENAGES_PRINC_2022_CSV_FR/DS_RP_MENAGES_PRINC_2022_data.csv', sep=';')
df_conjugality = pd.DataFrame(conjugality_csv)


# Suppression des lignes et colonnes inutiles
df_conjugality = df_conjugality[(df_conjugality['TIME_PERIOD'] == 2022) & (df_conjugality['GEO_OBJECT'] == 'COM') & (df_conjugality['RP_MEASURE'] == 'POP') & (df_conjugality['NOC'] == '_T') & (df_conjugality['OCS'] == '_T')]
df_conjugality = df_conjugality.drop(['FREQ', 'TIME_PERIOD', 'GEO_OBJECT', 'RP_MEASURE', 'OCS', 'NOC'], axis=1)


# Doublons et valeurs manquantes
print('Doublons et valeurs manquantes après suppression des données inutiles :')
handle_null_and_duplicates(df_conjugality)


# Premier aperçu du dataframe
display(df_conjugality)


# Conversion de la colonne GEO en str pour éviter les valeurs manquantes générées par les int
df_conjugality = convert_to_str(df_conjugality, ['GEO'])


# Récupération et application des métadonnées
conjugality_meta_csv = pd.read_csv('src_insee/DS_RP_MENAGES_PRINC_2022_CSV_FR/DS_RP_MENAGES_PRINC_2022_metadata.csv', sep=';')
df_conjugality_meta = pd.DataFrame(conjugality_meta_csv)

df_conjugality = replace_metadata(df_conjugality, df_conjugality_meta, ['AGE', 'CIVIL_STATUS', 'COUPLE'])
df_conjugality = add_cities_metadata(df_conjugality, df_conjugality_meta)


# Doublons et valeurs manquantes après récupération des métadonnées
print('Doublons et valeurs manquantes après récupération des communes :')
handle_null_and_duplicates(df_conjugality)


# Changement de nom pour les colonnes pas claires
df_conjugality.rename(columns={'OBS_VALUE': 'POPULATION'}, inplace=True)


# Aperçu des données après récupération des métadonnées
display(df_conjugality)


## Chargement dataset - Ménages

In [ ]:
# Chargement données + métadonnées
household_csv = pd.read_csv('src_insee/DS_RP_MENAGES_COMP_2022_CSV_FR/DS_RP_MENAGES_COMP_2022_data.csv', sep=';')
df_household = pd.DataFrame(household_csv)


# Suppression des lignes et colonnes inutiles
df_household = df_household[(df_household['TIME_PERIOD'] == 2022) & (df_household['GEO_OBJECT'] == 'COM') & (df_household['PREFPH'] == '_T') & (df_household['RP_MEASURE'] == 'DWELLINGS_POPSIZE')]
df_household = df_household.drop(['TIME_PERIOD', 'GEO_OBJECT', 'FREQ', 'OCS', 'PREFPH', 'RP_MEASURE'], axis=1)


# Doublons et valeurs manquantes
print('Doublons et valeurs manquantes après suppression des données inutiles :')
handle_null_and_duplicates(df_household)


# Premier aperçu des données
display(df_household)


# Conversion de la colonne GEO en str pour éviter les valeurs manquantes générées par les int
df_household = convert_to_str(df_household, ['GEO'])


# Récupération et application des métadonnées
household_meta_csv = pd.read_csv('src_insee/DS_RP_MENAGES_COMP_2022_CSV_FR/DS_RP_MENAGES_COMP_2022_metadata.csv', sep=';')
df_household_meta = pd.DataFrame(household_meta_csv)

df_household = replace_metadata(df_household, df_household_meta, ['PCS', 'TPH'])
df_household = add_cities_metadata(df_household, df_household_meta)


# Doublons et valeurs manquantes après récupération des métadonnées
print('Doublons et valeurs manquantes après récupération des communes :')
handle_null_and_duplicates(df_household)


# Changement de nom pour les colonnes pas claires
df_household.rename(columns={'OBS_VALUE': 'POPULATION', 'PCS': 'SOCIO_PROF_CAT', 'TPH': 'HOUSEHOLD_TYPE'}, inplace=True)


# Aperçu des données après récupération des métadonnées
display(df_household)

## Harmonisation des communes

In [ ]:
# Récupération des villes communes aux 3 dataframes
print('--- Dataset population')
set_codes_population = count_geo_codes(df_population)
print('\n--- Dataset conjugality')
set_codes_conjugality = count_geo_codes(df_conjugality)
print('\n--- Dataset ménages')
set_codes_household = count_geo_codes(df_household)

cities_to_keep = set_codes_population & set_codes_conjugality & set_codes_household
print('\n--- Nombre de communes présentes dans les 3 datasets :', len(cities_to_keep))


# Suppression des communes absentes du set cities_to_keep
df_population_final = df_population[df_population['CITY'].isin(cities_to_keep)].copy()
df_conjugality_final = df_conjugality[df_conjugality['CITY'].isin(cities_to_keep)].copy()
df_household_final = df_household[df_household['CITY'].isin(cities_to_keep)].copy()


# Vérification des communes
if df_population_final['CITY'].nunique() == df_conjugality_final['CITY'].nunique() == df_household_final['CITY'].nunique():
    print('\n--- DATAFRAMES HARMONISÉS')
else:
    print('\nÉCHEC DE L\'HARMONISATION')


print('\nDataset population :')
handle_null_and_duplicates(df_population_final)
print('\nDataset conjugality :')
handle_null_and_duplicates(df_conjugality_final)
print('\nDataset househould :')
handle_null_and_duplicates(df_household_final)

# Export des datasets
df_population_final.to_csv('src/population.csv', index=False, sep=';', encoding='utf-8')
df_conjugality_final.to_csv('src/conjugality.csv', index=False, sep=';', encoding='utf-8')
df_household_final.to_csv('src/household.csv', index=False, sep=';', encoding='utf-8')